
Objetivo
- Nesta etapa vamos transformar os dados prontos para análise em dados prontos para treinamento.

O que vamos fazer?
1. carregar o dataset validado
2. verificar tipos e valores ausentes
3. limpar colunas e converter tipos
4. separar features e alvo
5. dividir treino e teste
6. aplicar transformações para o modelo
7. salvar os dados preparados

Importante
- A preparação de dados é a etapa em que o modelo deixa de ver texto e valores inconsistentes para ver dados em formato útil.


In [38]:
# %%
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

import joblib

print("✓ Bibliotecas importadas com sucesso!")

✓ Bibliotecas importadas com sucesso!


In [39]:
# %%
print("=" * 60)
print("CONFIGURAÇÃO DO PROJETO")
print("=" * 60)

try:
    notebook_dir = Path.cwd()

    if notebook_dir.name == "notebooks":
        project_root = notebook_dir.parent
    else:
        project_root = notebook_dir

except Exception:
    project_root = Path.cwd()

# Diretórios
data_processed_dir = project_root / "data" / "processed"
artifacts_dir = project_root / "artifacts"
artifacts_tables_dir = artifacts_dir / "tables"
artifacts_models_dir = artifacts_dir / "models"

# Criar diretórios
artifacts_tables_dir.mkdir(
    parents=True,
    exist_ok=True
)

artifacts_models_dir.mkdir(
    parents=True,
    exist_ok=True
)

print(f"\nDiretório do projeto:")
print(project_root)

print("\nDiretório de dados:")
print(data_processed_dir)

print("\nDiretório de artefatos:")
print(artifacts_dir)

print("\n✓ Diretórios configurados!")

CONFIGURAÇÃO DO PROJETO

Diretório do projeto:
c:\Temp\telco-churn-ml

Diretório de dados:
c:\Temp\telco-churn-ml\data\processed

Diretório de artefatos:
c:\Temp\telco-churn-ml\artifacts

✓ Diretórios configurados!


In [40]:
# %%
print("=" * 60)
print("CARREGAMENTO DOS DADOS")
print("=" * 60)

input_file = (
    data_processed_dir /
    "01_raw_validated.csv"
)

print(f"\nArquivo esperado:")
print(input_file)

print(f"\nArquivo encontrado: {input_file.exists()}")

if not input_file.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {input_file}"
    )

df = pd.read_csv(input_file)

print("\n✓ Dataset carregado com sucesso!")

print(f"\nDimensões:")
print(f"  Linhas:  {df.shape[0]:,}")
print(f"  Colunas: {df.shape[1]}")

print("\nColunas:")
print(df.columns.tolist())

print("\nPrimeiras 5 linhas:")
display(df.head())

CARREGAMENTO DOS DADOS

Arquivo esperado:
c:\Temp\telco-churn-ml\data\processed\01_raw_validated.csv

Arquivo encontrado: True

✓ Dataset carregado com sucesso!

Dimensões:
  Linhas:  7,043
  Colunas: 21

Colunas:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Primeiras 5 linhas:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [41]:
# %%
print("=" * 60)
print("VERIFICAÇÃO INICIAL")
print("=" * 60)

print("\nTipos de dados:")
print(df.dtypes)

print("\nValores ausentes:")
print(df.isnull().sum())

print("\nDuplicatas:")
print(df.duplicated().sum())

print("\nDistribuição de Churn:")
print(df["Churn"].value_counts())

print("\n✓ Verificação inicial concluída.")

VERIFICAÇÃO INICIAL

Tipos de dados:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

Valores ausentes:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract    

In [42]:
# %%
print("=" * 60)
print("CONVERSÃO DE TOTALCHARGES")
print("=" * 60)

print(f"\nTipo antes da conversão:")
print(df["TotalCharges"].dtype)

# Converter para numérico
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

print(f"\nTipo depois da conversão:")
print(df["TotalCharges"].dtype)

print("\nValores ausentes após conversão:")
print(df["TotalCharges"].isnull().sum())

print("\n✓ TotalCharges convertido para numérico.")

CONVERSÃO DE TOTALCHARGES

Tipo antes da conversão:
object

Tipo depois da conversão:
float64

Valores ausentes após conversão:
11

✓ TotalCharges convertido para numérico.


In [43]:
# %%
print("=" * 60)
print("VERIFICAÇÃO DE VALORES AUSENTES")
print("=" * 60)

missing = df.isnull().sum()

missing_table = pd.DataFrame({
    "Valores_Ausentes": missing,
    "Percentual_%": (
        missing / len(df) * 100
    ).round(2)
})

missing_table = missing_table[
    missing_table["Valores_Ausentes"] > 0
]

if missing_table.empty:
    print("\n✓ Nenhum valor ausente encontrado.")
else:
    print("\nValores ausentes encontrados:")
    display(missing_table)

print("\n✓ Verificação concluída.")

VERIFICAÇÃO DE VALORES AUSENTES

Valores ausentes encontrados:


,Valores_Ausentes,Percentual_%
TotalCharges,11,0.16



✓ Verificação concluída.


In [44]:
# %%
print("=" * 60)
print("ENGENHARIA DE ATRIBUTOS (FEATURE ENGINEERING)")
print("=" * 60)

# 1. Quantidade de Serviços Contratados (Mede o engajamento)
# Clientes com muitos serviços atrelados são mais difíceis de cancelar
servicos = [
    'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]

# Verifica as colunas que existem no df para evitar erros
servicos_existentes = [col for col in servicos if col in df.columns]

df['TotalServices'] = df[servicos_existentes].apply(lambda x: (x == 'Yes').sum(), axis=1)

# 2. Perfil de Risco (Contrato Mensal + Fibra Ótica)
# Historicamente, clientes com Fibra Ótica (cara) e contrato mensal desistem muito rápido
if 'Contract' in df.columns and 'InternetService' in df.columns:
    df['HighRisk_Profile'] = (
        (df['Contract'] == 'Month-to-month') & 
        (df['InternetService'] == 'Fiber optic')
    ).astype(int)

# 3. Relação Gasto Real vs Cobrança
# Identifica se o cliente está pagando mais agora do que a média histórica dele (aumento de preço)
if 'tenure' in df.columns and 'TotalCharges' in df.columns and 'MonthlyCharges' in df.columns:
    # Evitar divisão por zero onde tenure é 0
    df['AvgRealMonthlyCharge'] = np.where(
        df['tenure'] > 0, 
        df['TotalCharges'] / df['tenure'], 
        df['MonthlyCharges']
    )
    
    # Diferença (Spike) de cobrança
    df['Charge_Spike'] = df['MonthlyCharges'] - df['AvgRealMonthlyCharge']

# 4. Agrupamento de Fidelidade (Tenure em anos)
if 'tenure' in df.columns:
    df['Tenure_Years'] = pd.cut(
        df['tenure'], 
        bins=[-1, 12, 24, 48, 60, 200], 
        labels=['0-1_Ano', '1-2_Anos', '2-4_Anos', '4-5_Anos', '+5_Anos']
    )
    # Converter para string/object para o Pipeline Categórico reconhecer corretamente
    df['Tenure_Years'] = df['Tenure_Years'].astype(str)

print("\nNovas features criadas com sucesso:")
print(" - TotalServices (Numérica)")
print(" - HighRisk_Profile (Numérica/Binária)")
print(" - Charge_Spike (Numérica)")
print(" - Tenure_Years (Categórica)")

ENGENHARIA DE ATRIBUTOS (FEATURE ENGINEERING)

Novas features criadas com sucesso:
 - TotalServices (Numérica)
 - HighRisk_Profile (Numérica/Binária)
 - Charge_Spike (Numérica)
 - Tenure_Years (Categórica)


In [45]:
# %%
print("=" * 60)
print("SEPARAÇÃO ENTRE VARIÁVEIS E ALVO")
print("=" * 60)

# Variável-alvo
target = "Churn"

# Separar variável-alvo
X = df.drop(columns=[target]).copy()

y = df[target].copy()

print("\nVariável-alvo:")
print(target)

print("\nDimensão de X:")
print(X.shape)

print("\nDimensão de y:")
print(y.shape)

print("\nDistribuição original de y:")
print(y.value_counts())

print("\n✓ X e y separados.")

SEPARAÇÃO ENTRE VARIÁVEIS E ALVO

Variável-alvo:
Churn

Dimensão de X:
(7043, 25)

Dimensão de y:
(7043,)

Distribuição original de y:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

✓ X e y separados.


In [46]:
# %%
print("=" * 60)
print("REMOÇÃO DO IDENTIFICADOR")
print("=" * 60)

if "customerID" in X.columns:

    print("\ncustomerID encontrado.")

    X = X.drop(
        columns=["customerID"]
    )

    print("✓ customerID removido.")

else:
    print("\n✓ customerID não está presente.")
    
print("\nDimensão atual de X:")
print(X.shape)

print("\nVariáveis restantes:")
print(X.columns.tolist())

REMOÇÃO DO IDENTIFICADOR

customerID encontrado.
✓ customerID removido.

Dimensão atual de X:
(7043, 24)

Variáveis restantes:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'TotalServices', 'HighRisk_Profile', 'AvgRealMonthlyCharge', 'Charge_Spike', 'Tenure_Years']


In [47]:
# %%
print("=" * 60)
print("CODIFICAÇÃO DA VARIÁVEL-ALVO")
print("=" * 60)

print("\nValores originais:")
print(y.value_counts())

# Converter Churn para 0/1
y = y.map({
    "No": 0,
    "Yes": 1
})

print("\nValores após codificação:")
print(y.value_counts())

print("\nTipo da variável:")
print(y.dtype)

print("\n✓ Churn convertido para variável binária.")

CODIFICAÇÃO DA VARIÁVEL-ALVO

Valores originais:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Valores após codificação:
Churn
0    5174
1    1869
Name: count, dtype: int64

Tipo da variável:
int64

✓ Churn convertido para variável binária.


In [48]:
# %%
print("=" * 60)
print("VALIDAÇÃO DA VARIÁVEL-ALVO")
print("=" * 60)

print("\nValores únicos:")
print(sorted(y.unique()))

print("\nValores ausentes:")
print(y.isnull().sum())

if set(y.dropna().unique()).issubset({0, 1}):
    print("\n✓ Churn está corretamente codificado como 0/1.")
else:
    print("\n⚠ Verificar codificação de Churn.")

VALIDAÇÃO DA VARIÁVEL-ALVO

Valores únicos:
[np.int64(0), np.int64(1)]

Valores ausentes:
0

✓ Churn está corretamente codificado como 0/1.


In [49]:
# %%
print("=" * 60)
print("SEPARAÇÃO TREINO E TESTE")
print("=" * 60)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nDimensões:")
print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test : {y_test.shape}")

print("\nDistribuição de Churn no treino:")
print(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nDistribuição de Churn no teste:")
print(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\n✓ Separação treino/teste concluída.")

SEPARAÇÃO TREINO E TESTE

Dimensões:
X_train: (5634, 24)
X_test : (1409, 24)
y_train: (5634,)
y_test : (1409,)

Distribuição de Churn no treino:
Churn
0    73.46
1    26.54
Name: proportion, dtype: float64

Distribuição de Churn no teste:
Churn
0    73.46
1    26.54
Name: proportion, dtype: float64

✓ Separação treino/teste concluída.


In [50]:
# %%
print("=" * 60)
print("VERIFICAÇÃO DO BALANCEAMENTO")
print("=" * 60)

train_balance = pd.DataFrame({
    "Quantidade": y_train.value_counts(),
    "Percentual_%": (
        y_train.value_counts(normalize=True) * 100
    ).round(2)
})

test_balance = pd.DataFrame({
    "Quantidade": y_test.value_counts(),
    "Percentual_%": (
        y_test.value_counts(normalize=True) * 100
    ).round(2)
})

print("\nTREINO")
print(train_balance)

print("\nTESTE")
print(test_balance)

print("\n✓ Balanceamento preservado por estratificação.")

VERIFICAÇÃO DO BALANCEAMENTO

TREINO
       Quantidade  Percentual_%
Churn                          
0            4139         73.46
1            1495         26.54

TESTE
       Quantidade  Percentual_%
Churn                          
0            1035         73.46
1             374         26.54

✓ Balanceamento preservado por estratificação.


In [51]:
# %%
print("=" * 60)
print("IDENTIFICAÇÃO DAS VARIÁVEIS")
print("=" * 60)

numeric_features = X_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("\nVariáveis numéricas:")
for col in numeric_features:
    print(f" - {col}")

print("\nVariáveis categóricas:")
for col in categorical_features:
    print(f" - {col}")

print("\nQuantidade:")
print(f"Numéricas:    {len(numeric_features)}")
print(f"Categóricas:  {len(categorical_features)}")

IDENTIFICAÇÃO DAS VARIÁVEIS

Variáveis numéricas:
 - SeniorCitizen
 - tenure
 - MonthlyCharges
 - TotalCharges
 - TotalServices
 - HighRisk_Profile
 - AvgRealMonthlyCharge
 - Charge_Spike

Variáveis categóricas:
 - gender
 - Partner
 - Dependents
 - PhoneService
 - MultipleLines
 - InternetService
 - OnlineSecurity
 - OnlineBackup
 - DeviceProtection
 - TechSupport
 - StreamingTV
 - StreamingMovies
 - Contract
 - PaperlessBilling
 - PaymentMethod
 - Tenure_Years

Quantidade:
Numéricas:    8
Categóricas:  16


In [52]:
# %%
print("=" * 60)
print("PIPELINE NUMÉRICO")
print("=" * 60)

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

print("\nPipeline numérico:")
print(numeric_pipeline)

print("\n✓ Pipeline numérico criado.")

PIPELINE NUMÉRICO

Pipeline numérico:
Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

✓ Pipeline numérico criado.


In [53]:
# %%
print("=" * 60)
print("PIPELINE CATEGÓRICO")
print("=" * 60)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

print("\nPipeline categórico:")
print(categorical_pipeline)

print("\n✓ Pipeline categórico criado.")

PIPELINE CATEGÓRICO

Pipeline categórico:
Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

✓ Pipeline categórico criado.


In [54]:
# %%
print("=" * 60)
print("CONSTRUÇÃO DO PRÉ-PROCESSADOR")
print("=" * 60)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop"
)

print("\nPré-processador:")
print(preprocessor)

print("\n✓ ColumnTransformer criado.")

CONSTRUÇÃO DO PRÉ-PROCESSADOR

Pré-processador:
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['SeniorCitizen', 'tenure', 'MonthlyCharges',
                                  'TotalCharges', 'TotalServices',
                                  'HighRisk_Profile', 'AvgRealMonthlyCharge',
                                  'Charge_Spike']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                spa

In [55]:
# %%
print("=" * 60)
print("AJUSTE DO PRÉ-PROCESSADOR")
print("=" * 60)

preprocessor.fit(X_train)

print("\n✓ Pré-processador ajustado utilizando apenas os dados de treino.")

AJUSTE DO PRÉ-PROCESSADOR

✓ Pré-processador ajustado utilizando apenas os dados de treino.


In [56]:
# %%
print("=" * 60)
print("TRANSFORMAÇÃO DOS DADOS")
print("=" * 60)

X_train_processed = preprocessor.transform(
    X_train
)

X_test_processed = preprocessor.transform(
    X_test
)

print("\nDimensões após transformação:")

print(
    f"X_train original:    {X_train.shape}"
)

print(
    f"X_train processado:   {X_train_processed.shape}"
)

print(
    f"X_test original:     {X_test.shape}"
)

print(
    f"X_test processado:    {X_test_processed.shape}"
)

print("\n✓ Dados transformados com sucesso.")

TRANSFORMAÇÃO DOS DADOS

Dimensões após transformação:
X_train original:    (5634, 24)
X_train processado:   (5634, 54)
X_test original:     (1409, 24)
X_test processado:    (1409, 54)

✓ Dados transformados com sucesso.


In [57]:
# %%
print("=" * 60)
print("NOMES DAS VARIÁVEIS APÓS ENCODING")
print("=" * 60)

feature_names = preprocessor.get_feature_names_out()

print(f"\nQuantidade total de features:")
print(len(feature_names))

print("\nPrimeiras 30 features:")

for feature in feature_names[:30]:
    print(f" - {feature}")

print("\n✓ Nomes das features obtidos.")

NOMES DAS VARIÁVEIS APÓS ENCODING

Quantidade total de features:
54

Primeiras 30 features:
 - num__SeniorCitizen
 - num__tenure
 - num__MonthlyCharges
 - num__TotalCharges
 - num__TotalServices
 - num__HighRisk_Profile
 - num__AvgRealMonthlyCharge
 - num__Charge_Spike
 - cat__gender_Female
 - cat__gender_Male
 - cat__Partner_No
 - cat__Partner_Yes
 - cat__Dependents_No
 - cat__Dependents_Yes
 - cat__PhoneService_No
 - cat__PhoneService_Yes
 - cat__MultipleLines_No
 - cat__MultipleLines_No phone service
 - cat__MultipleLines_Yes
 - cat__InternetService_DSL
 - cat__InternetService_Fiber optic
 - cat__InternetService_No
 - cat__OnlineSecurity_No
 - cat__OnlineSecurity_No internet service
 - cat__OnlineSecurity_Yes
 - cat__OnlineBackup_No
 - cat__OnlineBackup_No internet service
 - cat__OnlineBackup_Yes
 - cat__DeviceProtection_No
 - cat__DeviceProtection_No internet service

✓ Nomes das features obtidos.


In [58]:
# %%
print("=" * 60)
print("CRIAÇÃO DOS DATAFRAMES PROCESSADOS")
print("=" * 60)

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

print("\nX_train processado:")
display(X_train_processed_df.head())

print("\nX_test processado:")
display(X_test_processed_df.head())

print("\n✓ DataFrames processados criados.")

CRIAÇÃO DOS DATAFRAMES PROCESSADOS

X_train processado:


,num__SeniorCitizen,num__tenure,num__MonthlyCharges,num__TotalCharges,num__TotalServices,num__HighRisk_Profile,num__AvgRealMonthlyCharge,num__Charge_Spike,cat__gender_Female,cat__gender_Male,...,cat__PaperlessBilling_Yes,cat__PaymentMethod_Bank transfer (automatic),cat__PaymentMethod_Credit card (automatic),cat__PaymentMethod_Electronic check,cat__PaymentMethod_Mailed check,cat__Tenure_Years_+5_Anos,cat__Tenure_Years_0-1_Ano,cat__Tenure_Years_1-2_Anos,cat__Tenure_Years_2-4_Anos,cat__Tenure_Years_4-5_Anos
3738,-0.441773,0.102371,-0.521976,-0.263290,-0.184954,-0.659305,-0.539986,0.226917,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3151,-0.441773,-0.711743,0.337478,-0.504815,-0.667823,1.516749,0.391496,-0.639564,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4860,-0.441773,-0.793155,-0.809013,-0.751214,-0.184954,-0.659305,-0.646102,-1.867855,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
3867,-0.441773,-0.263980,0.284384,-0.173700,0.780784,-0.659305,0.276552,0.081601,1.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3810,-0.441773,-1.281624,-0.676279,-0.990851,-1.150692,-0.659305,-0.674608,0.003149,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0



X_test processado:


,num__SeniorCitizen,num__tenure,num__MonthlyCharges,num__TotalCharges,num__TotalServices,num__HighRisk_Profile,num__AvgRealMonthlyCharge,num__Charge_Spike,cat__gender_Female,cat__gender_Male,...,cat__PaperlessBilling_Yes,cat__PaymentMethod_Bank transfer (automatic),cat__PaymentMethod_Credit card (automatic),cat__PaymentMethod_Electronic check,cat__PaymentMethod_Mailed check,cat__Tenure_Years_+5_Anos,cat__Tenure_Years_0-1_Ano,cat__Tenure_Years_1-2_Anos,cat__Tenure_Years_2-4_Anos,cat__Tenure_Years_4-5_Anos
437,-0.441773,1.608483,1.629976,2.707614,2.229390,-0.659305,1.742949,-1.368443,0.0,1.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2280,2.263606,-0.996684,1.168725,-0.611506,0.780784,1.516749,1.609103,-5.161169,1.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2235,-0.441773,0.346606,0.445324,0.399490,1.263653,-0.659305,0.442847,0.013944,1.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4460,-0.441773,-0.589626,0.440347,-0.365546,-0.184954,1.516749,0.551220,-1.304300,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3761,-0.441773,1.608483,0.588013,1.588523,1.746522,-0.659305,0.571602,0.171258,1.0,0.0,...,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0



✓ DataFrames processados criados.


In [59]:
# %%
print("=" * 60)
print("VALIDAÇÃO DO PRÉ-PROCESSAMENTO")
print("=" * 60)

print("\nValores ausentes em X_train:")
print(
    X_train_processed_df.isnull().sum().sum()
)

print("\nValores ausentes em X_test:")
print(
    X_test_processed_df.isnull().sum().sum()
)

print("\nValores infinitos em X_train:")
print(
    np.isinf(X_train_processed_df).sum().sum()
)

print("\nValores infinitos em X_test:")
print(
    np.isinf(X_test_processed_df).sum().sum()
)

print("\nDimensões finais:")
print(
    f"X_train: {X_train_processed_df.shape}"
)

print(
    f"X_test : {X_test_processed_df.shape}"
)

print(
    f"y_train: {y_train.shape}"
)

print(
    f"y_test : {y_test.shape}"
)

print("\n✓ Validação concluída.")

VALIDAÇÃO DO PRÉ-PROCESSAMENTO

Valores ausentes em X_train:
0

Valores ausentes em X_test:
0

Valores infinitos em X_train:
0

Valores infinitos em X_test:
0

Dimensões finais:
X_train: (5634, 54)
X_test : (1409, 54)
y_train: (5634,)
y_test : (1409,)

✓ Validação concluída.


In [60]:
# %%
print("=" * 60)
print("VERIFICAÇÃO DA PADRONIZAÇÃO")
print("=" * 60)

numeric_feature_names = [
    feature
    for feature in feature_names
    if feature.startswith("num__")
]

if numeric_feature_names:

    numeric_means = (
        X_train_processed_df[numeric_feature_names]
        .mean()
        .round(3)
    )

    numeric_std = (
        X_train_processed_df[numeric_feature_names]
        .std()
        .round(3)
    )

    print("\nMédias das variáveis numéricas após StandardScaler:")
    print(numeric_means)

    print("\nDesvios padrão:")
    print(numeric_std)

else:
    print("\nNenhuma variável numérica encontrada.")

print("\n✓ Verificação da padronização concluída.")

VERIFICAÇÃO DA PADRONIZAÇÃO

Médias das variáveis numéricas após StandardScaler:
num__SeniorCitizen           0.0
num__tenure                 -0.0
num__MonthlyCharges         -0.0
num__TotalCharges           -0.0
num__TotalServices          -0.0
num__HighRisk_Profile        0.0
num__AvgRealMonthlyCharge   -0.0
num__Charge_Spike           -0.0
dtype: float64

Desvios padrão:
num__SeniorCitizen           1.0
num__tenure                  1.0
num__MonthlyCharges          1.0
num__TotalCharges            1.0
num__TotalServices           1.0
num__HighRisk_Profile        1.0
num__AvgRealMonthlyCharge    1.0
num__Charge_Spike            1.0
dtype: float64

✓ Verificação da padronização concluída.


In [61]:
# %%
print("=" * 60)
print("SALVANDO DADOS PROCESSADOS")
print("=" * 60)

# Caminhos
X_train_path = (
    artifacts_tables_dir /
    "X_train_processed.csv"
)

X_test_path = (
    artifacts_tables_dir /
    "X_test_processed.csv"
)

y_train_path = (
    artifacts_tables_dir /
    "y_train.csv"
)

y_test_path = (
    artifacts_tables_dir /
    "y_test.csv"
)

# Salvar
X_train_processed_df.to_csv(
    X_train_path,
    index=False
)

X_test_processed_df.to_csv(
    X_test_path,
    index=False
)

y_train.to_frame("Churn").to_csv(
    y_train_path,
    index=False
)

y_test.to_frame("Churn").to_csv(
    y_test_path,
    index=False
)

print("\n✓ X_train salvo em:")
print(X_train_path)

print("\n✓ X_test salvo em:")
print(X_test_path)

print("\n✓ y_train salvo em:")
print(y_train_path)

print("\n✓ y_test salvo em:")
print(y_test_path)

SALVANDO DADOS PROCESSADOS

✓ X_train salvo em:
c:\Temp\telco-churn-ml\artifacts\tables\X_train_processed.csv

✓ X_test salvo em:
c:\Temp\telco-churn-ml\artifacts\tables\X_test_processed.csv

✓ y_train salvo em:
c:\Temp\telco-churn-ml\artifacts\tables\y_train.csv

✓ y_test salvo em:
c:\Temp\telco-churn-ml\artifacts\tables\y_test.csv


In [62]:
# %%
print("=" * 60)
print("SALVANDO O PRÉ-PROCESSADOR")
print("=" * 60)

preprocessor_path = (
    artifacts_models_dir /
    "preprocessor.joblib"
)

joblib.dump(
    preprocessor,
    preprocessor_path
)

print("\n✓ Pré-processador salvo em:")
print(preprocessor_path)

SALVANDO O PRÉ-PROCESSADOR

✓ Pré-processador salvo em:
c:\Temp\telco-churn-ml\artifacts\models\preprocessor.joblib


In [63]:
# %%
print("=" * 60)
print("RESUMO DA ETAPA 3")
print("=" * 60)

print("\nDataset original:")
print(f"  {df.shape[0]:,} clientes")
print(f"  {df.shape[1]} colunas")

print("\nDivisão dos dados:")
print(f"  Treino: {X_train.shape[0]:,} clientes")
print(f"  Teste : {X_test.shape[0]:,} clientes")

print("\nVariáveis:")
print(f"  Numéricas:   {len(numeric_features)}")
print(f"  Categóricas: {len(categorical_features)}")

print("\nFeatures após One-Hot Encoding:")
print(f"  {len(feature_names)}")

print("\nDados processados:")
print(f"  X_train: {X_train_processed_df.shape}")
print(f"  X_test : {X_test_processed_df.shape}")

print("\nArquivos gerados:")
print("  - X_train_processed.csv")
print("  - X_test_processed.csv")
print("  - y_train.csv")
print("  - y_test.csv")
print("  - preprocessor.joblib")

print("\n✓ ETAPA 3 CONCLUÍDA COM SUCESSO!")

RESUMO DA ETAPA 3

Dataset original:
  7,043 clientes
  26 colunas

Divisão dos dados:
  Treino: 5,634 clientes
  Teste : 1,409 clientes

Variáveis:
  Numéricas:   8
  Categóricas: 16

Features após One-Hot Encoding:
  54

Dados processados:
  X_train: (5634, 54)
  X_test : (1409, 54)

Arquivos gerados:
  - X_train_processed.csv
  - X_test_processed.csv
  - y_train.csv
  - y_test.csv
  - preprocessor.joblib

✓ ETAPA 3 CONCLUÍDA COM SUCESSO!
